In [1]:
# bge-m3模型
from milvus_model.hybrid import BGEM3EmbeddingFunction

/Users/chan/projects/Itcast_qa_system/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
m3_path = "../rag_qa/models/bge-m3"
embedding_function = BGEM3EmbeddingFunction(model_name_or_path=m3_path, use_fp16=False, device='cpu')

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 49556.49it/s]


In [3]:
embed = embedding_function(["黑马JAVA", "黑马大模型", "黑马大模型"])
embed

{'dense': [array([-0.03068057,  0.00156495, -0.04899885, ...,  0.04231707,
          0.01612839,  0.0713326 ], shape=(1024,), dtype=float32),
  array([-0.01287245, -0.01234254, -0.06556936, ...,  0.04092151,
         -0.01335396,  0.05632111], shape=(1024,), dtype=float32),
  array([-0.01287245, -0.01234254, -0.06556936, ...,  0.04092151,
         -0.01335396,  0.05632111], shape=(1024,), dtype=float32)],
 'sparse': <Compressed Sparse Row sparse array of dtype 'float64'
 	with 14 stored elements and shape (3, 250002)>}

In [6]:
embed["dense"]

[array([-0.03068057,  0.00156495, -0.04899885, ...,  0.04231707,
         0.01612839,  0.0713326 ], shape=(1024,), dtype=float32),
 array([-0.01287245, -0.01234254, -0.06556936, ...,  0.04092151,
        -0.01335396,  0.05632111], shape=(1024,), dtype=float32),
 array([-0.01287245, -0.01234254, -0.06556936, ...,  0.04092151,
        -0.01335396,  0.05632111], shape=(1024,), dtype=float32)]

In [7]:
row = embed["sparse"][0]

# try:
#     # 新版本 milvus-model 使用 coo_array 格式
#     row = embed["sparse"][0]
#     if hasattr(row, 'col'):  # coo_array 格式
#         indices = row.col
#         values = row.data
#     else:  # csr_matrix 格式
#         indices = row.indices
#         values = row.data
# except Exception as e:
#     # 兼容旧版本 milvus-model
#     row = embed["sparse"].getrow(0)
#     indices = row.indices
#     values = row.data15

print(row.col)
print(row.data)
# print(indices)
# print(values)

[     6   5958   7320 189950]
[0.00417884 0.24346864 0.15046994 0.28743052]


In [7]:
row.col

array([     6,   5958,   7320, 189950])

In [8]:
row.data

array([0.00417884, 0.24346864, 0.15046994, 0.28743052])

In [8]:
# reranker 模型
from sentence_transformers import CrossEncoder

In [9]:
# reranker_path = "../rag_qa/models/bge-reranker-large"
reranker_path = "/Users/chan/projects/models/bge-reranker-v2-m3"
reranker = CrossEncoder(reranker_path)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 8011.13it/s]


In [10]:
reranker

CrossEncoder(
  (0): Transformer({'transformer_task': 'sequence-classification', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'logits'}}, 'module_output_name': 'scores', 'architecture': 'XLMRobertaForSequenceClassification'})
)

In [13]:
queries = "机器学习"
documents = [
    "深度学习是机器学习的一个分支。",
    "今天的天气很好。",
    "机器学习是一种让计算机从数据中自动学习规律的方法。",
]
# 输入内容是文本对
pairs = [[queries, doc] for doc in documents]
print(pairs)


[['机器学习', '深度学习是机器学习的一个分支。'], ['机器学习', '今天的天气很好。'], ['机器学习', '机器学习是一种让计算机从数据中自动学习规律的方法。']]


In [14]:
scores = reranker.predict(pairs)
print(f'scores-->{scores}')


# 输出排序结果
for doc, score in sorted(zip(documents, scores), key=lambda x: x[1], reverse=True):
    print(f"{score:.4f} - {doc}")

scores-->[6.0132432e-01 1.6151498e-05 9.9906605e-01]
0.9991 - 机器学习是一种让计算机从数据中自动学习规律的方法。
0.6013 - 深度学习是机器学习的一个分支。
0.0000 - 今天的天气很好。


In [18]:
# reranker.predict(["我喜欢这个苹果", "他觉得apple很好！"])
reranker.predict(["我喜欢这个苹果", "我不喜欢这个苹果！"])

np.float32(0.0009145845)